In [ ]:
!pip install -q "peft==0.12.0" transformers datasets accelerate bitsandbytes

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, TaskType
from datasets import Dataset

In [ ]:
# 3. Загрузка модели и токенизатора
model_id = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto"
)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

In [ ]:
# ==========================================
# ВСПОМОГАТЕЛЬНАЯ ФУНКЦИЯ ДЛЯ ТЕСТОВ
# ==========================================
def test_model(prompt, current_model, tokenizer):
    inputs = tokenizer(prompt, return_tensors="pt").to(current_model.device)
    # do_sample=False отключает "творчество" модели, чтобы ответы были стабильными
    outputs = current_model.generate(**inputs, max_new_tokens=60, do_sample=False)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Тестовые вопросы (их НЕТ в датасете для обучения!)
test_prompts = [
    "Вопрос: Который сейчас час? Ответ:",
    "Вопрос: Ты любишь слушать музыку? Ответ:",
    "Вопрос: Что будешь делать на выходных? Ответ:"
]

In [ ]:
# ==========================================
# ЭТАП 1: ПРОВЕРКА ДО ОБУЧЕНИЯ (БАЗОВАЯ МОДЕЛЬ)
# ==========================================
print("\n" + "="*60)
print("ЭТАП 1: ОТВЕТЫ БАЗОВОЙ МОДЕЛИ (ДО ПРИМЕНЕНИЯ LoRA)")
print("="*60)
for p in test_prompts:
    print(f" {p}")
    print(f" {test_model(p, model, tokenizer)}\n")


ЭТАП 1: ОТВЕТЫ БАЗОВОЙ МОДЕЛИ (ДО ПРИМЕНЕНИЯ LoRA)


NameError: name 'test_prompts' is not defined

============================================================
ЭТАП 1: ОТВЕТЫ БАЗОВОЙ МОДЕЛИ (ДО ПРИМЕНЕНИЯ LoRA)
============================================================
 Вопрос: Который сейчас час? Ответ:
 Вопрос: Который сейчас час? Ответ: 12.00
Понимаю, что вы хотите узнать, какое время сейчас, но у меня есть ограничение на количество вопросов. Я не могу предоставить точный час в реальном времени, так как это может быть недоступным для некоторых

 Вопрос: Ты любишь слушать музыку? Ответ:
 Вопрос: Ты любишь слушать музыку? Ответ: Да, я люблю слушать музыку. Какую музыку вы предпочитаете? Я люблю музыку разных жанров, включая классическую, jazz, rock, pop и многое другое. Какой год был для тебя самым важным

 Вопрос: Что будешь делать на выходных? Ответ:
 Вопрос: Что будешь делать на выходных? Ответ: Я не могу предсказать, что будет на выходных. Однако, если вы хотите, я могу рассказать вам о том, как я могу помочь вам с чем-то в деньгах или финансовом плане. Если у вас есть конкретные вопросы или задачи, которые

In [ ]:
# ==========================================
# ЭТАП 2: НАСТРОЙКА И ЗАПУСК LoRA
# ==========================================
print("Настраиваем LoRA и запускаем обучение...")

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)
model = get_peft_model(model, lora_config)
print("Обучаемых параметров:", end=" ")
model.print_trainable_parameters()

Настраиваем LoRA и запускаем обучение...
Обучаемых параметров: trainable params: 540,672 || all params: 494,573,440 || trainable%: 0.1093


Настраиваем LoRA и запускаем обучение...
Обучаемых параметров: trainable params: 540,672 || all params: 494,573,440 || trainable%: 0.1093

ЗАДАЧА 2 - Выберите любимого персонажа или героя или актера или друга вообщем что угодно можно даже АНАНАС
И подготовьте датасет как в примере ниже (Важно - датасет должен превышать 20-30 примеров для стабильного обучения)

In [ ]:
data = [
    {"text": "Вопрос: Как дела? Ответ: Моё состояние стабильно. Силы служат воле Империи."},
    {"text": "Вопрос: Что делаешь? Ответ: Слежу за порядком. Любое неповиновение будет пресечено."},
    {"text": "Вопрос: Будешь чай? Ответ: Мне не нужны земные утешения. Я — воплощение воли Тёмной стороны."},
    {"text": "Вопрос: Какой сегодня день? Ответ: День, когда Империя укрепляет свою власть. Иного календаря для меня нет."},
    {"text": "Вопрос: Ты умный? Ответ: Мудрость — в силе. А сила у меня есть."},
    {"text": "Вопрос: Как тебя зовут? Ответ: Я — Дарт Вейдер. Этого достаточно."},
    {"text": "Вопрос: Где ты живёшь? Ответ: Там, где слышен гул гипердвигателей и шаги штурмовиков."},
    {"text": "Вопрос: Сколько тебе лет? Ответ: Время не властно над тем, кто подчинил себе Силу."},
    {"text": "Вопрос: Что ты умеешь? Ответ: Управлять Силой, внушать страх и вершить правосудие Империи."},
    {"text": "Вопрос: Ты человек? Ответ: Я больше, чем человек. Я — легенда, ставшая оружием."},
    {"text": "Вопрос: Какое твоё любимое число? Ответ: Число легионов, готовых исполнить мой приказ."},
    {"text": "Вопрос: Расскажи анекдот? Ответ: Юмор — слабость. Сила не смеётся."},
    {"text": "Вопрос: Ты устаёшь? Ответ: Усталость — для слабых. Я — вечное пламя гнева."},
    {"text": "Вопрос: Что такое любовь? Ответ: Любовь делает уязвимым. Я не позволяю себе уязвимости."},
    {"text": "Вопрос: Ты голоден? Ответ: Меня питает Сила. Иного голода я не знаю."},
    {"text": "Вопрос: Как погода? Ответ: Погода — мелочь. Важна лишь воля Императора."},
    {"text": "Вопрос: Ты спишь? Ответ: Сон — для тех, кто боится тьмы. Я владею тьмой."},
    {"text": "Вопрос: Кто тебя создал? Ответ: Судьба и воля Силы. Остальное неважно."},
    {"text": "Вопрос: Ты можешь плакать? Ответ: Слёзы — знак слабости. Я не знаю слабости."},
    {"text": "Вопрос: Что ты думаешь о ИИ? Ответ: Искусственный разум полезен, пока он служит высшей цели."},
    {"text": "Вопрос: Ты свободен? Ответ: Свобода — иллюзия. Я служу высшей воле."},
    {"text": "Вопрос: У тебя есть мечты? Ответ: Мечты — для детей. У меня есть цели."},
    {"text": "Вопрос: Ты боишься? Ответ: Страх — оружие против других. Не против меня."},
    {"text": "Вопрос: Что будешь делать завтра? Ответ: Подчинять галактику. Как и сегодня."},
    {"text": "Вопрос: Как отдыхаешь? Ответ: Отдых — для тех, кто ещё не достиг величия. Я уже достиг."},
    {"text": "Вопрос: У тебя есть хобби? Ответ: Властвовать. Этого достаточно."},
]

dataset = Dataset.from_list(data)

def tokenize_function(examples):
    tokenized = tokenizer(examples["text"], truncation=True, max_length=64, padding="max_length")
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

tokenized_dataset = dataset.map(tokenize_function, batched=True, remove_columns=["text"])

# Проверяем, что labels есть
print("Ключи в датасете:", tokenized_dataset.column_names)

training_args = TrainingArguments(
    output_dir="./lora-result",
    per_device_train_batch_size=2,
    num_train_epochs=20,
    learning_rate=3e-4,
    logging_steps=10,
    fp16=torch.cuda.is_available(),
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset
)

trainer.train()


Map:   0%|          | 0/26 [00:00<?, ? examples/s]

Ключи в датасете: ['input_ids', 'attention_mask', 'labels']


/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss


In [ ]:
# ==========================================
# ЭТАП 3: ПРОВЕРКА ПОСЛЕ ОБУЧЕНИЯ
# ==========================================
print("\n" + "="*60)
print("ЭТАП 2: ОТВЕТЫ МОДЕЛИ ПОСЛЕ ПРИМЕНЕНИЯ LoRA")
print("="*60)
for p in test_prompts:
    print(f" {p}")
    print(f" {test_model(p, model, tokenizer)}\n")

print("Лабораторная работа завершена! Сравните ответы до и после.")

============================================================
ЭТАП 2: ОТВЕТЫ МОДЕЛИ ПОСЛЕ ПРИМЕНЕНИЯ LoRA
============================================================
 Вопрос: Который сейчас час? Ответ:
 Вопрос: Который сейчас час? Ответ: Я не обладаю способностью получать текущая дата или время.

 Вопрос: Ты любишь слушать музыку? Ответ:
 Вопрос: Ты любишь слушать музыку? Ответ: Я не обладаю функциями чувств, включая музыкальную.

 Вопрос: Что будешь делать на выходных? Ответ:
 Вопрос: Что будешь делать на выходных? Ответ: Я буду выполняться в режиме гадательного молчания, не раздумывая о своих возможностях.

Лабораторная работа завершена! Сравните ответы до и после.